# San Antonio bicyclist crash analysis

This notebook analyzes TxDOT CRIS records for pedalcyclists killed or suspected seriously injured. The current reporting window runs from 2024 through Sept. 1, 2026, matching the available CRIS records.

The person-level filters are `3 - PEDALCYCLIST` and `K - FATAL INJURY` **OR** `A - SUSPECTED SERIOUS INJURY`.

In [ ]:
from pathlib import Path
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
DATA = ROOT / 'data'
RAW = DATA / 'raw'
BOUNDARIES = DATA / 'boundaries'
OUT = ROOT / 'outputs'
BOUNDARIES.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

REPORTING_START = 2024
REPORTING_END = 2026
REPORTING_WINDOW = 'Jan. 1, 2024–Sept. 1, 2026'

# Helpers used for crash-level audits and aggregation. CRIS stores one row per
# affected person, so a crash can have multiple values for the same field.
def nonblank_values(series):
    values = set()
    for value in series:
        if pd.notna(value):
            text = str(value).strip()
            if text and text.upper() not in {'NAN', 'NONE', 'NO DATA'}:
                values.add(text)
    return sorted(values)

def first_non_null(series):
    series = series.dropna()
    return series.iloc[0] if len(series) else pd.NA

def union_values(series):
    return ' | '.join(nonblank_values(series))

def audit_fields(frame, id_col, columns):
    rows = []
    for record_id, group in frame.groupby(id_col):
        for column in columns:
            values = nonblank_values(group[column])
            if len(values) > 1:
                rows.append({id_col: record_id, 'field': column,
                             'distinct_values': ' | '.join(values),
                             'n_distinct': len(values)})
    return pd.DataFrame(rows, columns=[id_col, 'field', 'distinct_values', 'n_distinct'])


In [ ]:
# Load and filter the CRIS export. The export has 12 metadata rows before the header.
csv_path = RAW / 'myexport_final.csv'
raw = pd.read_csv(csv_path, skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type'] == '3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['Crash Year'] = pd.to_numeric(target['Crash Year'], errors='coerce').astype('Int64')
target['Latitude_num'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['Longitude_num'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['affected'] = 1
print('Filtered people:', len(target))
print('Unique crashes:', target['Crash ID'].nunique())
print(target.groupby(['City', 'Crash Year']).agg(people=('affected','sum'), serious_injuries=('serious_injury','sum'), deaths=('death','sum'), crashes=('Crash ID','nunique')).tail(12))

In [ ]:
# San Antonio trend: people, not crashes, are the outcome counts.
sa = target[target['City'] == 'SAN ANTONIO']
annual_sa = sa.groupby('Crash Year').agg(people=('affected','sum'), serious_injuries=('serious_injury','sum'), deaths=('death','sum'), crashes=('Crash ID','nunique')).reset_index()
annual_sa.to_csv(OUT / 'san_antonio_annual.csv', index=False)
annual_sa

In [ ]:
# Compare Texas cities with at least 250,000 residents in the same reporting window.
# The 2024 ACS 1-year population estimate is used as the denominator for all
# crashes from Jan. 1, 2024 through Sept. 1, 2026 because a 2026 estimate is
# not yet available. Rates are cumulative, not annualized.
# Paste the key when prompted; it will not appear in notebook output or be saved.
from getpass import getpass
census_key = getpass('Paste your Census API key: ')
if not census_key.strip():
    raise RuntimeError('A Census API key is required for this cell.')
census_url = 'https://api.census.gov/data/2024/acs/acs1'
census_response = requests.get(census_url, params={'get':'NAME,B01003_001E,B08301_001E,B08301_018E', 'for':'place:*', 'in':'state:48', 'key':census_key}, timeout=120)
census_response.raise_for_status()
if not census_response.text.lstrip().startswith('['):
    raise RuntimeError(f'Census API returned non-JSON: {census_response.text[:200]}')
census_rows = census_response.json()
census = pd.DataFrame(census_rows[1:], columns=census_rows[0])
census['population'] = pd.to_numeric(census['B01003_001E'])
census['workers'] = pd.to_numeric(census['B08301_001E'])
census['bike_commuters'] = pd.to_numeric(census['B08301_018E'])
census['bike_commute_pct'] = census['bike_commuters'] / census['workers'] * 100
census['City'] = (census['NAME'].str.replace(', Texas', '', regex=False)
                 .str.replace(r'\s+(city|town|village)$', '', case=False, regex=True).str.upper())
eligible_places = census[census['population'] >= 250000].copy()
texas_cities = (target[target['Crash Year'].between(REPORTING_START, REPORTING_END) & target['City'].isin(eligible_places['City'])]
           .groupby('City', as_index=False)
           .agg(crashes=('Crash ID','nunique'), affected_bicyclists=('affected','sum'), serious_injuries=('serious_injury','sum'), deaths=('death','sum')))
texas_cities = eligible_places[['City','population','workers','bike_commuters','bike_commute_pct']].merge(texas_cities, on='City', how='left')
count_cols = ['crashes','affected_bicyclists','serious_injuries','deaths']
texas_cities[count_cols] = texas_cities[count_cols].fillna(0).astype(int)
texas_cities['city'] = texas_cities['City'].str.title()
texas_cities['population_year'] = 2024
texas_cities['population_note'] = '2024 ACS 1-year estimate; denominator for cumulative ' + REPORTING_WINDOW + '; rates not annualized'
texas_cities['affected_rate_per_million'] = texas_cities['affected_bicyclists'] / texas_cities['population'] * 1000000
texas_cities['death_rate_per_million'] = texas_cities['deaths'] / texas_cities['population'] * 1000000
texas_cities = texas_cities.sort_values('death_rate_per_million', ascending=False)
texas_cities.to_csv(OUT / 'texas_cities_comparison_2024_2026.csv', index=False)
texas_cities_with_death = texas_cities[texas_cities['deaths'] >= 1].copy()
texas_cities_with_death.insert(0, 'death_rate_rank', range(1, len(texas_cities_with_death) + 1))
texas_cities_with_death.to_csv(OUT / 'texas_cities_with_death_comparison_2024_2026.csv', index=False)
print('Cities included: population >= 250,000.')
print('Population denominator: 2024 ACS 1-year estimate; rates are cumulative for ' + REPORTING_WINDOW + ' and are not annualized.')
display(texas_cities_with_death[['death_rate_rank','city','population','population_year','bike_commute_pct','crashes','affected_bicyclists','serious_injuries','deaths','affected_rate_per_million','death_rate_per_million']])


## What San Antonio's severe-injury and fatal crashes have in common

This section is descriptive, not causal. Intersection categories are crash-level and remain separate: `INTERSECTION`, `INTERSECTION RELATED`, `DRIVEWAY ACCESS` and `NON INTERSECTION` are not collapsed into one category. Helmet status is person-level. CRIS contributing factors can overlap and are police-recorded codes, not an independent determination of fault.

In [ ]:
sa_people = target[(target['City'] == 'SAN ANTONIO') & target['Crash Year'].between(REPORTING_START, REPORTING_END)].copy()

# Audit every field that will be used at crash level. This catches cases where
# person-level rows for one Crash ID disagree before any deduplication/counting.
audit_columns = [
    'Intersection Related', 'At Intersection Flag',
    'Street Name', 'Street Number', 'Intersecting Street Name',
    'Latitude_num', 'Longitude_num', 'Speed Limit',
    'Contributing Factor 1', 'Contributing Factor 2', 'Contributing Factor 3',
    'Possible Contributing Factor 1', 'Possible Contributing Factor 2'
]
crash_consistency = audit_fields(sa_people, 'Crash ID', audit_columns)
crash_consistency.to_csv(OUT / 'crash_row_consistency_audit_2024_2026.csv', index=False)
print('Crash-field consistency audit rows:', len(crash_consistency))
if len(crash_consistency) == 0:
    print('Crash-field consistency audit: No discrepancies found.')
if len(crash_consistency):
    display(crash_consistency.head(25))

# Keep one row per crash for crash-level counts, while unioning contributing
# factor values across all people in a crash instead of silently keeping the
# first person's factors.
sa_crashes = (sa_people.groupby('Crash ID', as_index=False)
              .agg(**{
                  'Intersection Related': ('Intersection Related', first_non_null),
                  'At Intersection Flag': ('At Intersection Flag', first_non_null),
                  'Street Name': ('Street Name', first_non_null),
                  'Street Number': ('Street Number', first_non_null),
                  'Intersecting Street Name': ('Intersecting Street Name', first_non_null),
                  'Contributing Factor 1': ('Contributing Factor 1', union_values),
                  'Contributing Factor 2': ('Contributing Factor 2', union_values),
                  'Contributing Factor 3': ('Contributing Factor 3', union_values),
                  'Possible Contributing Factor 1': ('Possible Contributing Factor 1', union_values),
                  'Possible Contributing Factor 2': ('Possible Contributing Factor 2', union_values),
                  'deaths': ('death', 'sum'),
                  'serious_injuries': ('serious_injury', 'sum'),
              }))
commonality_rows = []

intersection_categories = sa_crashes['Intersection Related'].value_counts(dropna=False)
for category, count in intersection_categories.items():
    commonality_rows.append({'topic':'Intersection context', 'category':str(category), 'unit':'crashes', 'count':int(count), 'denominator':int(len(sa_crashes)), 'share':count/len(sa_crashes)})

helmet = sa_people['Person Helmet'].value_counts(dropna=False)
for category, count in helmet.items():
    commonality_rows.append({'topic':'Helmet status', 'category':str(category), 'unit':'bicyclists', 'count':int(count), 'denominator':int(len(sa_people)), 'share':count/len(sa_people)})

factor_cols = ['Contributing Factor 1','Contributing Factor 2','Contributing Factor 3','Possible Contributing Factor 1','Possible Contributing Factor 2']
factor_text = sa_crashes[factor_cols].fillna('').astype(str).agg(' | '.join, axis=1)
factor_groups = {'Failed to yield':'FAILED TO YIELD', 'Driver inattention':'DRIVER INATTENTION', 'Disregard stop sign/light':'DISREGARD STOP', 'Speed-related code':'SPEED', 'Alcohol/intoxication code':'ALCOHOL|INTOXICATED', 'Drug code':'DRUG'}
for label, pattern in factor_groups.items():
    count = factor_text.str.contains(pattern, case=False, regex=True).sum()
    commonality_rows.append({'topic':'Contributing-factor code', 'category':label, 'unit':'crashes', 'count':int(count), 'denominator':int(len(sa_crashes)), 'share':count/len(sa_crashes)})

commonalities = pd.DataFrame(commonality_rows)
commonalities.to_csv(OUT / 'commonalities.csv', index=False)
commonalities

# Simple street and intersection rankings from the CRIS location fields.
sa_locations = (sa_people.groupby('Crash ID', as_index=False)
                .agg(year=('Crash Year','first'), street=('Street Name',first_non_null),
                     street_number=('Street Number',first_non_null), intersecting_street=('Intersecting Street Name',first_non_null),
                     deaths=('death','sum'), serious_injuries=('serious_injury','sum')))
for col in ['street','intersecting_street']:
    sa_locations[col] = sa_locations[col].fillna('').astype(str).str.strip()
street_hotspots = (sa_locations[sa_locations['street'] != '']
                  .groupby('street', as_index=False)
                  .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'))
                  .sort_values(['crashes','deaths'], ascending=False))
street_hotspots.to_csv(OUT / 'street_hotspots.csv', index=False)
sa_locations['intersection'] = np.where((sa_locations['street'] != '') & (sa_locations['intersecting_street'] != ''), sa_locations['street'] + ' & ' + sa_locations['intersecting_street'], '')
sa_locations.loc[sa_locations['street'] == sa_locations['intersecting_street'], 'intersection'] = ''
intersection_hotspots = (sa_locations[sa_locations['intersection'] != '']
                         .groupby('intersection', as_index=False)
                         .agg(crashes=('Crash ID','nunique'), deaths=('deaths','sum'), serious_injuries=('serious_injuries','sum'))
                         .sort_values(['crashes','deaths'], ascending=False))
intersection_hotspots.to_csv(OUT / 'intersection_hotspots.csv', index=False)
print('Top streets')
display(street_hotspots.head(15))
print('Top intersections')
display(intersection_hotspots.head(15))


## Driver-factor outcomes

This table reports the bicyclist deaths and suspected serious injuries associated with police-reported driver codes in the 2024–Sept. 1, 2026 window.

In [ ]:
# Simple verification table: driver codes and bicyclist outcomes.
current_people = target[(target['City'] == 'SAN ANTONIO') & target['Crash Year'].between(2024, 2026)].copy()
current_ids = set(current_people['Crash ID'])
outcomes = (current_people.groupby('Crash ID', as_index=False)
            .agg(deaths=('death','sum'), serious_injuries=('serious_injury','sum')))
drivers = raw[(raw['Crash ID'].isin(current_ids)) & (raw['Person Type'] == '1 - DRIVER')].copy()
factor_cols = ['Contributing Factor 1','Contributing Factor 2','Contributing Factor 3','Possible Contributing Factor 1','Possible Contributing Factor 2']
driver_text = drivers[factor_cols].fillna('').astype(str).agg(' | '.join, axis=1).str.upper()
driver_text = driver_text + ' | ' + drivers['Driver Alcohol Result'].fillna('').astype(str).str.upper() + ' | ' + drivers['Driver Drug Test Result'].fillna('').astype(str).str.upper()
factor_patterns = {
    'Driver inattention': 'INATTENTION',
    'Failure to yield': 'FAILED TO YIELD',
    'Speed-related code': 'SPEED|OVERLIMIT',
    'Alcohol or drug indicator': 'INTOXICATED|ALCOHOL|DRUG|POSITIVE',
    'Cell-phone use': 'CELL/MOBILE|PHONE|DEVICE',
}
rows = []
for factor, pattern in factor_patterns.items():
    factor_ids = set(drivers.loc[driver_text.str.contains(pattern, regex=True), 'Crash ID'])
    matched = outcomes[outcomes['Crash ID'].isin(factor_ids)]
    rows.append({'factor': factor, 'crashes': len(factor_ids), 'deaths': int(matched['deaths'].sum()), 'serious_injuries': int(matched['serious_injuries'].sum()), 'affected_bicyclists': int(matched['deaths'].sum() + matched['serious_injuries'].sum())})
factor_outcomes = pd.DataFrame(rows)
factor_outcomes.to_csv(OUT / 'driver_factors_outcomes_2024_2026.csv', index=False)
factor_outcomes

### Coverage warning

The uploaded export does not provide usable hit-and-run values, and it does not include a reliable narrative-level determination of fault. Those should not be presented as findings from this file.

In [ ]:
# Download boundary files only when needed.
tract_zip = BOUNDARIES / 'tl_2020_48_tract.zip'
tract_url = 'https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_48_tract.zip'
if not tract_zip.exists():
    tract_zip.write_bytes(requests.get(tract_url, timeout=120).content)
tract_dir = BOUNDARIES / 'tracts'
if not tract_dir.exists():
    tract_dir.mkdir()
    with zipfile.ZipFile(tract_zip) as z:
        z.extractall(tract_dir)
tracts = gpd.read_file(tract_dir / 'tl_2020_48_tract.shp', columns=['GEOID','NAME','COUNTYFP','geometry'])
tracts = tracts[tracts['COUNTYFP'] == '029'].to_crs(4326)

# Aggregate locations once per crash, but audit location fields first.
geo_audit_columns = ['Latitude_num', 'Longitude_num', 'Street Name', 'Street Number', 'Intersecting Street Name']
sa_geo_people = target[target['City'] == 'SAN ANTONIO'].copy()
geo_consistency = audit_fields(sa_geo_people, 'Crash ID', geo_audit_columns)
geo_consistency.to_csv(OUT / 'geo_field_consistency_audit_2024_2026.csv', index=False)
print('Geographic-field consistency audit rows:', len(geo_consistency))
if len(geo_consistency) == 0:
    print('Geographic-field consistency audit: No discrepancies found.')
if len(geo_consistency):
    display(geo_consistency.head(25))

crashes = (sa_geo_people.groupby('Crash ID', as_index=False)
           .agg(year=('Crash Year', first_non_null),
                latitude=('Latitude_num', first_non_null),
                longitude=('Longitude_num', first_non_null),
                serious_injuries=('serious_injury','sum'),
                deaths=('death','sum')))
crashes_before_coordinates = len(crashes)
missing_coordinate_mask = crashes[['latitude','longitude']].isna().any(axis=1)
missing_coordinate_crashes = int(missing_coordinate_mask.sum())
crashes = crashes.loc[~missing_coordinate_mask].copy()
print(f'Dropped {missing_coordinate_crashes} of {crashes_before_coordinates} unique San Antonio crashes ({missing_coordinate_crashes / crashes_before_coordinates:.1%}) from geographic analyses because latitude or longitude was missing.')
points = gpd.GeoDataFrame(crashes, geometry=gpd.points_from_xy(crashes['longitude'], crashes['latitude']), crs=4326)
tract_join = gpd.sjoin(points, tracts[['GEOID','NAME','geometry']], how='left', predicate='within')
tract_counts = tract_join[tract_join['year'].between(REPORTING_START, REPORTING_END)].groupby(['GEOID','NAME'], as_index=False).agg(serious_injuries=('serious_injuries','sum'), deaths=('deaths','sum'), crashes=('Crash ID','nunique'))
tract_counts['affected_bicyclists'] = tract_counts['serious_injuries'] + tract_counts['deaths']
tract_counts = tract_counts.sort_values(['affected_bicyclists','deaths'], ascending=False)
tract_counts.to_csv(OUT / 'census_tract_hotspots.csv', index=False)
tract_counts.head(15)


In [ ]:
# City Council districts from the official City of San Antonio ArcGIS service.
# Requesting outSR=4326 avoids the export endpoint's asynchronous download behavior.
council_url = 'https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/CouncilDistricts/FeatureServer/2/query?where=1%3D1&outFields=District&returnGeometry=true&outSR=4326&f=geojson'
council_file = BOUNDARIES / 'council_districts.geojson'
if not council_file.exists():
    council_file.write_bytes(requests.get(council_url, timeout=120).content)
council = gpd.read_file(council_file).to_crs(4326)
district_join = gpd.sjoin(points, council[['District','geometry']], how='left', predicate='within')
districts = district_join[district_join['year'].between(REPORTING_START, REPORTING_END)].groupby('District', as_index=False).agg(serious_injuries=('serious_injuries','sum'), deaths=('deaths','sum'), crashes=('Crash ID','nunique'))
districts['affected_bicyclists'] = districts['serious_injuries'] + districts['deaths']
districts = districts.sort_values(['affected_bicyclists','deaths'], ascending=False)
districts.to_csv(OUT / 'council_districts.csv', index=False)
districts


## Official bicycle High Injury Network corridors

This is the main location analysis for the story. It uses the City of San Antonio's official Bicycle High Injury Network corridors rather than an improvised hotspot rule. The city identified 22 corridors totaling 17.1 miles. This notebook compares the city's 2019–2023 study period with a 2024–Sept. 1, 2026 update window. CRIS points are matched within 150 feet of each corridor, and fatal crashes and suspected serious-injury crashes are counted separately. The update window runs through Sept. 1, 2026, matching the latest date in the CRIS export.

In [ ]:
# Download the City's official Bicycle HIN corridor layer.
hin_url = ('https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/'
           'SS4A_HIN_Dashboard_Data/FeatureServer/1/query?'
           'where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson')
hin_file = BOUNDARIES / 'bicycle_hin_corridors.geojson'
if not hin_file.exists():
    response = requests.get(hin_url, timeout=120)
    response.raise_for_status()
    hin_file.write_bytes(response.content)
hin = gpd.read_file(hin_file).to_crs(4326)
hin = hin.rename(columns={'Name': 'corridor'})

# Rebuild one row per qualifying bicyclist crash, including 2026.
hin_people = target[(target['City'] == 'SAN ANTONIO') &
                    target['Crash Year'].between(2019, REPORTING_END)].copy()
hin_geo_audit = audit_fields(hin_people, 'Crash ID', ['Latitude_num', 'Longitude_num'])
hin_geo_audit.to_csv(OUT / 'hin_coordinate_consistency_audit_2019_2026.csv', index=False)
hin_crashes = (hin_people.groupby('Crash ID', as_index=False)
              .agg(year=('Crash Year', first_non_null),
                   latitude=('Latitude_num', first_non_null),
                   longitude=('Longitude_num', first_non_null),
                   deaths=('death', 'sum'),
                   serious_injuries=('serious_injury', 'sum')))
hin_before_coordinates = len(hin_crashes)
hin_missing_coordinate_mask = hin_crashes[['latitude', 'longitude']].isna().any(axis=1)
hin_missing_coordinates = int(hin_missing_coordinate_mask.sum())
hin_crashes_with_coordinates = hin_crashes.loc[~hin_missing_coordinate_mask].copy()
print(f'Dropped {hin_missing_coordinates} of {hin_before_coordinates} unique HIN-analysis crashes ({hin_missing_coordinates / hin_before_coordinates:.1%}) from spatial matching because latitude or longitude was missing.')
if len(hin_geo_audit) == 0:
    print('HIN coordinate audit: No discrepancies found.')
hin_points = gpd.GeoDataFrame(
    hin_crashes_with_coordinates,
    geometry=gpd.points_from_xy(hin_crashes_with_coordinates['longitude'], hin_crashes_with_coordinates['latitude']),
    crs=4326
)

# The buffer makes the matching rule explicit and avoids counting a point
# only when it falls exactly on the centerline geometry.
hin_projected = hin.to_crs(2279).copy()
hin_projected['geometry'] = hin_projected.geometry.buffer(150)
hin_matches = gpd.sjoin(hin_points.to_crs(2279),
                        hin_projected[['bicycle_hin_id', 'corridor', 'Miles', 'geometry']],
                        how='inner', predicate='within')
# Spatial joins can return repeated rows for the same crash/corridor pair.
# This removes those repeated matches, but preserves one crash matched to each
# different HIN corridor.
hin_matches_before_dedup = len(hin_matches)
hin_matches = hin_matches.drop_duplicates(['Crash ID', 'bicycle_hin_id']).copy()
hin_matches_dropped = hin_matches_before_dedup - len(hin_matches)
print(f'HIN spatial join produced {hin_matches_before_dedup} rows; removed {hin_matches_dropped} duplicate crash/corridor matches.')

def summarize_hin(frame):
    return (frame.groupby(['bicycle_hin_id', 'corridor', 'Miles'], as_index=False)
            .agg(crashes=('Crash ID', 'nunique'),
                 deaths=('deaths', 'sum'),
                 serious_injuries=('serious_injuries', 'sum'),
                 first_year=('year', 'min'),
                 last_year=('year', 'max'))
            .sort_values(['crashes', 'deaths'], ascending=False))

hin_2019_2023 = summarize_hin(hin_matches[hin_matches['year'].between(2019, 2023)])
hin_2024_2026 = summarize_hin(hin_matches[hin_matches['year'].between(REPORTING_START, REPORTING_END)])
hin_2019_2023.to_csv(OUT / 'bicycle_hin_corridors_2019_2023.csv', index=False)
hin_2024_2026.to_csv(OUT / 'bicycle_hin_corridors_2024_2026.csv', index=False)

# The current reporting-window result needed for the story.
south_general = hin_matches[hin_matches['corridor'].str.upper().eq('S GENERAL MCMULLEN')]
south_general.groupby('year', as_index=False).agg(
    crashes=('Crash ID', 'nunique'), deaths=('deaths', 'sum'),
    serious_injuries=('serious_injuries', 'sum')
)

display(hin_2019_2023)
display(hin_2024_2026)


In [ ]:
# Editor-ready charts and Datawrapper-ready CSVs.
trend = annual_sa.copy()
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(trend['Crash Year'], trend['serious_injuries'], marker='o', color='#3478a4', label='Suspected serious injuries')
ax.plot(trend['Crash Year'], trend['deaths'], marker='o', color='#e67e22', label='Deaths')
ax.set_xlabel('Crash year')
ax.set_ylabel('Bicyclists')
ax.set_title('San Antonio bicyclists killed or seriously injured, 2016–Sept. 1, 2026')
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / 'san_antonio_annual_trend.png', dpi=200)
trend.to_csv(OUT / 'datawrapper_sa_annual_trend.csv', index=False)
plt.show()

# Top 10 death-rate cities, showing injuries and deaths separately.
top10 = texas_cities_with_death.head(10).copy()
top10['serious_injury_rate_per_million'] = top10['serious_injuries'] / top10['population'] * 1000000
top10_rates = top10[['city','serious_injury_rate_per_million','death_rate_per_million']].sort_values('death_rate_per_million')
top10_rates.to_csv(OUT / 'datawrapper_top10_city_rates.csv', index=False)
fig, ax = plt.subplots(figsize=(9, 5.8))
y = np.arange(len(top10_rates))
height = 0.36
ax.barh(y - height/2, top10_rates['serious_injury_rate_per_million'], height, color='#3478a4', label='Serious injuries')
ax.barh(y + height/2, top10_rates['death_rate_per_million'], height, color='#e67e22', label='Deaths')
ax.set_yticks(y, top10_rates['city'])
ax.set_xlabel('Bicyclists per 1 million residents')
ax.set_title('Highest bicyclist death rates among Texas cities with a death')
ax.legend(frameon=False)
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.text(0.01, 0.01, 'Rates use 2024 ACS 1-year population estimates as the denominator; cumulative Jan. 1, 2024–Sept. 1, 2026, not annualized.', fontsize=8)
fig.savefig(OUT / 'texas_top10_injury_death_rates.png', dpi=200)
plt.show()

# Bike commuting share versus bicyclist death rate.
scatter = texas_cities_with_death[['city','bike_commute_pct','death_rate_per_million','deaths','population']].dropna().copy()
scatter.to_csv(OUT / 'datawrapper_bike_commute_vs_death_rate.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(scatter['bike_commute_pct'], scatter['death_rate_per_million'], color='#5b8db8', alpha=.8)
for _, row in scatter.iterrows():
    ax.annotate(row['city'], (row['bike_commute_pct'], row['death_rate_per_million']), xytext=(4, 3), textcoords='offset points', fontsize=8)
ax.set_xlabel('Workers who commute by bicycle (%)')
ax.set_ylabel('Bicyclist deaths per 1 million residents')
ax.set_title('Bike commuting and bicyclist death rates in Texas cities')
fig.tight_layout()
fig.savefig(OUT / 'bike_commute_vs_death_rate.png', dpi=200)
plt.show()